# 13 — Query Decomposition RAG

> **Tier 3 | Query Handling**

## The Problem

Complex questions contain multiple implicit sub-questions that a single retrieval pass misses:

```
"How do greenhouse gas emissions affect both ocean acidity and polar ice extent?"
  → sub-Q1: How do GHG emissions affect ocean acidity?
  → sub-Q2: How do GHG emissions affect polar ice extent?
  → synthesis: Combine findings into a unified answer
```

A single embedding of the full question retrieves passages that are only partially relevant.
Decomposing ensures each concept gets its own focused retrieval pass.

## Solution

1. **Decompose** — LLM breaks the complex question into N focused sub-questions
2. **Retrieve** — vector search per sub-question (independent top-K)
3. **Deduplicate** — merge hit lists, drop repeated chunk IDs
4. **Synthesize** — LLM answers the original question using all retrieved passages

## Flow Diagram

```mermaid
flowchart TD
    Q(["❓ Complex Question"])
    Q --> DECOMP["🧠 LLM Decomposer\nClaude Sonnet 4.6\n→ sub-Q1 … sub-QN"]

    subgraph RETRIEVAL ["🔍  Per-Sub-Question Retrieval"]
        direction LR
        SQ1["sub-Q1\n→ top-K hits"] 
        SQ2["sub-Q2\n→ top-K hits"]
        SQN["sub-QN\n→ top-K hits"]
    end

    DECOMP --> SQ1
    DECOMP --> SQ2
    DECOMP --> SQN

    SQ1 --> MERGE["🔀 Merge & Deduplicate\n(by chunk ID)"]
    SQ2 --> MERGE
    SQN --> MERGE

    MERGE --> SYNTH["🟠 Strands Agent\nSynthesise answer\nfrom all passages"]
    SYNTH --> ANS(["✅ Final attributed answer"])

    style RETRIEVAL fill:#dbeafe,stroke:#3b82f6,color:#1e3a5f
    style MERGE     fill:#dcfce7,stroke:#16a34a,color:#14532d
    style SYNTH     fill:#ffedd5,stroke:#f97316,color:#7c2d12
```

In [1]:
import subprocess, sys
packages = ["boto3","qdrant-client","opensearch-py","requests-aws4auth",
            "strands-agents","pypdf","python-dotenv"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + packages)
print("All packages ready.")

All packages ready.


## Step 2 — Imports & Configuration

In [2]:
import os, json, time, uuid, re
from typing import List, Dict, Optional

import boto3, pypdf
from dotenv import load_dotenv
from strands import Agent
from strands.models.bedrock import BedrockModel
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    Filter, FieldCondition, MatchValue
)

load_dotenv(r"C:\Users\Administrator\RAG\.env", override=True)

AWS_REGION      = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
EMBEDDING_MODEL = "amazon.titan-embed-text-v2:0"
LLM_MODEL       = "us.anthropic.claude-sonnet-4-6"

QDRANT_URL     = os.getenv("QDRANT_URL", "")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "")
OPENSEARCH_URL = os.getenv("OPENSEARCH_ENDPOINT", "")

COLLECTION_NAME = "query_decomposition_13"
EMBEDDING_DIM   = 1024
TOP_K           = 4    # hits per sub-question
CHUNK_SIZE      = 500
CHUNK_OVERLAP   = 50
PDF_PATH        = r"C:\Users\Administrator\RAG\data\climate.pdf"

print(f"Collection : {COLLECTION_NAME}")
print(f"PDF exists : {os.path.exists(PDF_PATH)}")
print(f"AWS region : {AWS_REGION}")
print(f"Key ID     : {os.getenv('AWS_ACCESS_KEY_ID','NOT SET')[:12]}...")

Collection : query_decomposition_13
PDF exists : True
AWS region : us-east-1
Key ID     : ASIA4KPWEP6M...


## Step 3 — Vector Store

In [3]:
class VectorStore:
    def __init__(self, collection_name, qdrant_url='', qdrant_api_key='',
                 opensearch_url='', region='us-east-1'):
        self.name = collection_name
        self._backend = None
        if qdrant_url:
            try:
                kw = {'url': qdrant_url}
                if qdrant_api_key: kw['api_key'] = qdrant_api_key
                self._qdrant = QdrantClient(**kw)
                self._qdrant.get_collections()
                self._backend = 'qdrant_cloud'
                print(f'Qdrant Cloud: {qdrant_url}')
                return
            except Exception as e:
                print(f'Qdrant Cloud unavailable ({e}), trying next...')
        if opensearch_url:
            try:
                from opensearchpy import OpenSearch, RequestsHttpConnection, AWSV4SignerAuth
                creds = boto3.Session().get_credentials()
                auth  = AWSV4SignerAuth(creds, region, 'aoss')
                host  = opensearch_url.replace('https://','').replace('http://','')
                self._os = OpenSearch(
                    hosts=[{'host': host, 'port': 443}],
                    http_auth=auth, use_ssl=True, verify_certs=True,
                    connection_class=RequestsHttpConnection, timeout=30)
                self._os.info()
                self._backend = 'opensearch'
                print(f'OpenSearch: {opensearch_url}')
                return
            except Exception as e:
                print(f'OpenSearch unavailable ({e}), falling back...')
        self._qdrant  = QdrantClient(':memory:')
        self._backend = 'qdrant_memory'
        print('Using Qdrant in-memory')

    def create_collection(self, dim=1024, force_recreate=False):
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            exists = self.name in [c.name for c in self._qdrant.get_collections().collections]
            if exists and force_recreate:
                self._qdrant.delete_collection(self.name); exists = False
            if not exists:
                self._qdrant.create_collection(
                    self.name, vectors_config=VectorParams(size=dim, distance=Distance.COSINE))
                print(f'Created "{self.name}" (dim={dim})')
            else:
                print(f'"{self.name}" already exists — skipping recreate')

    def upsert(self, docs: List[Dict]) -> int:
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            pts = [PointStruct(id=str(uuid.uuid4()), vector=d['embedding'],
                               payload={'text': d['text'], 'metadata': d.get('metadata', {})})
                   for d in docs]
            self._qdrant.upsert(collection_name=self.name, points=pts)
            return len(pts)

    def search(self, qvec: List[float], top_k: int = 5,
               query_filter=None) -> List[Dict]:
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            resp = self._qdrant.query_points(
                collection_name=self.name, query=qvec, limit=top_k,
                query_filter=query_filter, with_payload=True)
            return [{'text': p.payload.get('text', ''),
                     'metadata': p.payload.get('metadata', {}),
                     'score': p.score, 'id': str(p.id)}
                    for p in resp.points]
        return []

    def count(self) -> int:
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            return self._qdrant.get_collection(self.name).points_count or 0
        return 0

    def delete_collection(self):
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            self._qdrant.delete_collection(self.name)
        print(f'Deleted "{self.name}"')

print("VectorStore defined.")

VectorStore defined.


## Step 4 — Bedrock Helpers

In [4]:
bedrock_rt = boto3.client('bedrock-runtime', region_name=AWS_REGION)

def embed_text(text: str) -> List[float]:
    body = json.dumps({"inputText": text, "dimensions": EMBEDDING_DIM, "normalize": True})
    resp = bedrock_rt.invoke_model(
        modelId=EMBEDDING_MODEL, body=body,
        contentType="application/json", accept="application/json")
    return json.loads(resp["body"].read())["embedding"]

def embed_batch(texts: List[str], label: str = '') -> List[List[float]]:
    out = []
    for i, t in enumerate(texts):
        out.append(embed_text(t))
        if (i + 1) % 20 == 0:
            print(f'  {label} {i+1}/{len(texts)}')
        time.sleep(0.04)
    return out

_model = BedrockModel(model_id=LLM_MODEL, region_name=AWS_REGION)

def llm(prompt: str) -> str:
    return str(Agent(model=_model, system_prompt="You are a helpful assistant.")(prompt))

test_emb = embed_text("query decomposition test")
print(f"Embedding OK — dim={len(test_emb)}")
print("Bedrock LLM ready.")

Embedding OK — dim=1024
Bedrock LLM ready.


## Step 5 — Connect & Index climate.pdf

In [5]:
def recursive_split(text: str, chunk_size: int, chunk_overlap: int) -> List[str]:
    separators = ["\n\n", "\n", ". ", " ", ""]
    def _split(text, seps):
        sep, new_seps = '', []
        for i, s in enumerate(seps):
            if s == '' or s in text:
                sep, new_seps = s, seps[i+1:]; break
        parts = text.split(sep) if sep != '' else list(text)
        good = []
        for part in parts:
            if len(part) <= chunk_size: good.append(part)
            elif new_seps: good.extend(_split(part, new_seps))
            else:
                for k in range(0, len(part), chunk_size - chunk_overlap):
                    good.append(part[k:k+chunk_size])
        chunks, cur_pieces, cur_len = [], [], 0
        for piece in good:
            p = piece.strip()
            if not p: continue
            addition = len(sep) + len(p) if cur_pieces else len(p)
            if cur_len + addition <= chunk_size:
                cur_pieces.append(p); cur_len += addition
            else:
                if cur_pieces: chunks.append(sep.join(cur_pieces))
                ov, ovl = [], 0
                for prev in reversed(cur_pieces):
                    if ovl + len(prev) + len(sep) <= chunk_overlap:
                        ov.insert(0, prev); ovl += len(prev) + len(sep)
                    else: break
                cur_pieces = ov + [p]
                cur_len = sum(len(x) + len(sep) for x in cur_pieces)
        if cur_pieces: chunks.append(sep.join(cur_pieces))
        return [c for c in chunks if c.strip()]
    return _split(text, separators)

# Connect
vs = VectorStore(
    collection_name=COLLECTION_NAME,
    qdrant_url=QDRANT_URL,
    qdrant_api_key=QDRANT_API_KEY,
    opensearch_url=OPENSEARCH_URL,
    region=AWS_REGION
)
print(f"Backend: {vs._backend}")
vs.create_collection(dim=EMBEDDING_DIM, force_recreate=True)

# Load & chunk PDF
reader    = pypdf.PdfReader(PDF_PATH)
full_text = ''
page_map  = []
for pg_idx, page in enumerate(reader.pages):
    pg_text = (page.extract_text() or '') + '\n\n'
    page_map.extend([pg_idx + 1] * len(pg_text))
    full_text += pg_text

chunks = recursive_split(full_text, CHUNK_SIZE, CHUNK_OVERLAP)
print(f"PDF pages: {len(reader.pages)}  |  Chunks: {len(chunks)}")

# Embed & index
embs = embed_batch(chunks, label='[index]')
docs = []
for i, (chunk, emb) in enumerate(zip(chunks, embs)):
    char_offset = full_text.find(chunk[:40])
    page_num = page_map[min(char_offset, len(page_map)-1)] if char_offset >= 0 else 1
    docs.append({
        'text'     : chunk,
        'embedding': emb,
        'metadata' : {'chunk_idx': i, 'page_num': page_num, 'char_count': len(chunk)}
    })
vs.upsert(docs)
print(f"Indexed {vs.count()} vectors into '{COLLECTION_NAME}'")

Qdrant Cloud: https://2210ff1c-7c49-4fec-91f4-01662586299c.us-east-1-1.aws.cloud.qdrant.io
Backend: qdrant_cloud
Created "query_decomposition_13" (dim=1024)
PDF pages: 13  |  Chunks: 66
  [index] 20/66
  [index] 40/66
  [index] 60/66
Indexed 66 vectors into 'query_decomposition_13'


## Step 6 — Query Decomposer

The LLM receives the complex question and returns a JSON list of focused sub-questions.

In [6]:
def decompose_query(question: str, max_subs: int = 4) -> List[str]:
    prompt = f"""Break the following complex question into {max_subs} or fewer focused sub-questions.
Each sub-question should be self-contained and answerable from a document corpus.
Return ONLY a JSON array of strings — no explanation, no markdown fences.

Complex question: {question}

JSON array:"""
    raw = llm(prompt).strip()
    # Strip markdown fences if present
    raw = re.sub(r'^```[a-z]*\n?', '', raw, flags=re.MULTILINE)
    raw = re.sub(r'```$', '', raw, flags=re.MULTILINE)
    try:
        subs = json.loads(raw.strip())
        if isinstance(subs, list):
            return [s for s in subs if isinstance(s, str) and s.strip()]
    except json.JSONDecodeError:
        # Fallback: extract quoted strings
        subs = re.findall(r'"([^"]+)"', raw)
        if subs:
            return subs
    return [question]  # fallback: treat original as single sub-question

# Smoke test
test_q = "How do greenhouse gas emissions affect ocean acidity and polar ice extent?"
subs   = decompose_query(test_q)
print(f"Original: {test_q}")
print(f"Sub-questions ({len(subs)}):")
for i, s in enumerate(subs, 1):
    print(f"  {i}. {s}")

["What are the primary greenhouse gases emitted by human activities?", "How do increased greenhouse gas emissions lead to rising ocean acidity?", "How do greenhouse gas emissions contribute to the melting of polar ice and reduced ice extent?", "What is the relationship between ocean acidity changes and polar ice extent?"]Original: How do greenhouse gas emissions affect ocean acidity and polar ice extent?
Sub-questions (4):
  1. What are the primary greenhouse gases emitted by human activities?
  2. How do increased greenhouse gas emissions lead to rising ocean acidity?
  3. How do greenhouse gas emissions contribute to the melting of polar ice and reduced ice extent?
  4. What is the relationship between ocean acidity changes and polar ice extent?


## Step 7 — Decomposed Retrieval Pipeline

In [7]:
def rag_decomposed(question: str, top_k: int = TOP_K, verbose: bool = True) -> Dict:
    t0 = time.time()

    # 1. Decompose
    sub_questions = decompose_query(question)

    # 2. Retrieve per sub-question
    all_hits: List[Dict] = []
    seen_ids = set()
    sub_hits_counts = []
    for sq in sub_questions:
        qvec = embed_text(sq)
        hits = vs.search(qvec, top_k=top_k)
        fresh = [h for h in hits if h['id'] not in seen_ids]
        for h in fresh:
            seen_ids.add(h['id'])
        all_hits.extend(fresh)
        sub_hits_counts.append(len(fresh))

    # 3. Sort merged hits by score
    all_hits.sort(key=lambda h: h['score'], reverse=True)

    # 4. Synthesise
    context_parts = []
    for h in all_hits:
        m = h['metadata']
        src = f"[p.{m.get('page_num','?')}]"
        context_parts.append(f"{src}\n{h['text']}")

    context = '\n\n'.join(context_parts)
    synth_prompt = (
        "Using ONLY the passages below, answer the question thoroughly. "
        "Cite page numbers (e.g. [p.3]) for each key fact.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"
    )
    answer = llm(synth_prompt)
    latency = (time.time() - t0) * 1000

    if verbose:
        print(f"\nQ: {question}")
        print(f"Sub-questions ({len(sub_questions)}):")
        for i, (sq, n) in enumerate(zip(sub_questions, sub_hits_counts), 1):
            print(f"  {i}. [{n} unique hits] {sq}")
        print(f"Total unique passages: {len(all_hits)}  |  Latency: {latency:.0f}ms")
        print(f"\nAnswer:\n{answer}")
        print('-' * 70)

    return {
        'question'     : question,
        'sub_questions': sub_questions,
        'hits'         : all_hits,
        'answer'       : answer,
        'latency_ms'   : latency,
    }

# Test
rag_decomposed(
    "How do greenhouse gas emissions affect ocean acidity and polar ice extent?"
)

["What are the primary greenhouse gases emitted by human activities?", "How do greenhouse gas emissions contribute to ocean acidification?", "What is the relationship between rising global temperatures from greenhouse gases and polar ice melting?", "How does the reduction in polar ice extent affect ocean acidity levels?"]Based on the provided passages, I **cannot answer** this question. None of the passages contain any information about greenhouse gas emissions, ocean acidity, or polar ice extent. The passages focus exclusively on topics such as weather data collection and transmission, weather forecasting methods, satellites, and meteorological organizations.

To answer your question accurately, different source materials addressing climate science and environmental impacts would be needed.
Q: How do greenhouse gas emissions affect ocean acidity and polar ice extent?
Sub-questions (4):
  1. [4 unique hits] What are the primary greenhouse gases emitted by human activities?
  2. [4 uniq

{'question': 'How do greenhouse gas emissions affect ocean acidity and polar ice extent?',
 'sub_questions': ['What are the primary greenhouse gases emitted by human activities?',
  'How do greenhouse gas emissions contribute to ocean acidification?',
  'What is the relationship between rising global temperatures from greenhouse gases and polar ice melting?',
  'How does the reduction in polar ice extent affect ocean acidity levels?'],
 'hits': [{'text': 'Melbourne in Australia and Moscow in Russia. This enables in the dissemination of daily\nweather forecasts and early and reliable warnings of high-impact weather and climate events.\nAfter the processes of recording, collection and transmission of data comes the work of\ncompilation and analysis of data which is done by climatological experts. Computers are used in\nthe final analysis work and various models are in use by the experts which we would learn in the',
   'metadata': {'chunk_idx': 17, 'page_num': 5, 'char_count': 461},
   '

## Step 8 — Baseline Comparison: Simple vs Decomposed

Run the same complex questions through both strategies to show that decomposition retrieves more relevant, diverse passages.

In [8]:
def rag_simple(question: str, top_k: int = TOP_K) -> Dict:
    qvec = embed_text(question)
    hits = vs.search(qvec, top_k=top_k)
    context = '\n\n'.join(f"[p.{h['metadata'].get('page_num','?')}]\n{h['text']}" for h in hits)
    answer  = llm(
        f"Answer using ONLY the context. Cite page numbers.\n\nContext:\n{context}\n\nQ: {question}\n\nA:"
    )
    return {'hits': hits, 'answer': answer}

eval_questions = [
    "What causes sea level rise and what are the projected impacts on coastal populations?",
    "How do feedback loops in the climate system amplify or dampen warming?",
    "What evidence exists for changes in precipitation patterns due to climate change?",
]

print(f"{'Question':<55}  {'Simple hits':>11}  {'Decomp hits':>11}  {'Sub-Qs':>6}")
print('-' * 90)

for q in eval_questions:
    r_simple = rag_simple(q)
    r_decomp = rag_decomposed(q, verbose=False)
    simple_pages = sorted({h['metadata'].get('page_num','?') for h in r_simple['hits']})
    decomp_pages = sorted({h['metadata'].get('page_num','?') for h in r_decomp['hits']})
    print(f"{q[:54]:<55}  {str(simple_pages):>11}  {str(decomp_pages):>11}  {len(r_decomp['sub_questions']):>6}")

Question                                                 Simple hits  Decomp hits  Sub-Qs
------------------------------------------------------------------------------------------
The provided context does not contain any information about the causes of sea level rise or projected impacts on coastal populations. The context only discusses weather forecasting methods, the IMD, and related meteorological topics.["What are the primary causes of sea level rise?", "How much has sea level risen historically and what are the projections for future sea level rise?", "Which coastal regions and populations are most vulnerable to sea level rise?", "What are the projected impacts of sea level rise on coastal communities and infrastructure?"]Based on the provided passages, **this question cannot be answered**. The passages do not contain any information about the causes of sea level rise or its projected impacts on coastal populations.

The passages focus entirely on different topics, including:
-

## Step 9 — Deep Multi-Part Question

Show the full decomposed answer for a complex synthesis question.

In [9]:
complex_q = (
    "What are the primary drivers of global temperature increase, "
    "how do they interact with ocean and ice systems, "
    "and what are the consequences for extreme weather events?"
)
rag_decomposed(complex_q, top_k=3)

["What are the primary drivers of global temperature increase?", "How does global warming interact with ocean systems, including sea level rise and ocean heat content?", "How does global warming affect ice systems such as glaciers, ice sheets, and sea ice?", "What are the consequences of global temperature increase for extreme weather events?"]Based on the provided passages, this question **cannot be fully answered**. The passages focus primarily on **weather analysis, forecasting methods, and data collection systems**, and do not contain information about the primary drivers of global temperature increase, interactions with ocean and ice systems, or the specific consequences for extreme weather events in the context of climate change.

However, the passages do contain a few **tangentially related points** that can be noted:

- The passages mention that meteorological agencies collect **meteorological, climatological, hydrological, and oceanographic data** from a vast global network, i

{'question': 'What are the primary drivers of global temperature increase, how do they interact with ocean and ice systems, and what are the consequences for extreme weather events?',
 'sub_questions': ['What are the primary drivers of global temperature increase?',
  'How does global warming interact with ocean systems, including sea level rise and ocean heat content?',
  'How does global warming affect ice systems such as glaciers, ice sheets, and sea ice?',
  'What are the consequences of global temperature increase for extreme weather events?'],
 'hits': [{'text': 'related to certain weather sensitive activities like agriculture, water resource management,\nindustries, oil exploration, shipping, aviation etc. It has a tremendous role in giving pre-warning\nin case of extreme weather events like cyclone, floods, thunderstorms, heat and cold waves etc.\nThe information is disseminated to the disaster management agencies and general public so as to\nprevent the loss of life and proper

## Step 10 — Summary

| Component | Implementation |
|-----------|---------------|
| Decomposer | Claude Sonnet 4.6 → JSON array of sub-questions |
| Retrieval | Top-K vector search per sub-question |
| Deduplication | Set of chunk IDs — each passage retrieved at most once |
| Synthesis | Single LLM call over all unique passages with page citations |
| Vector DB | Qdrant Cloud → OpenSearch → in-memory |
| Embeddings | Amazon Bedrock Titan V2 (1024-dim) |

## Key trade-offs

| Concern | Notes |
|---------|-------|
| Latency | 1 LLM call (decompose) + N embed calls + 1 LLM call (synthesise) |
| Token cost | Larger context window in synthesis step |
| Quality gain | Wider page coverage; each sub-concept gets dedicated retrieval |
| Fallback | If JSON parse fails, original question used as single sub-question |

### Next: **14 — Step-Back Prompting**

In [ ]:
# vs.delete_collection()  # uncomment to clean up
print(f"Collection '{COLLECTION_NAME}' in {vs._backend} — {vs.count()} vectors")
print("\nDone. Give the go-ahead for notebook 14.")